**Part 7 · RAG Mastery** · step 35 of 37
[← 34 Interview questions](34_interview_questions.md) · [🏠 Roadmap](../README.md) · [36 Lab: hybrid + rerank →](36_lab_hybrid_rerank_pipeline.ipynb)

# 35 · Lab 🔌: Search Engine from Scratch (BM25, RRF, MMR, Metrics)

Runs **offline**, no API key, no model download. You build the core of a search engine with plain Python,
so nothing in RAG feels like magic anymore.

**What you'll build:**
1. An analyzer (tokenize → lowercase → stop words → tiny stemmer)
2. An **inverted index**
3. **BM25** from scratch, and check it against the worked example in [28](28_bm25_and_elasticsearch.md)
4. See **k1 saturation** and **b length normalisation** with numbers
5. **RRF** fusion from scratch
6. Retrieval metrics: **Hit@k, Recall@k, Precision@k, MRR, nDCG**
7. **MMR** for diversity
8. A first evaluation of BM25 on a labelled set, and where keyword search fails

In [ ]:
import math
import re
from collections import Counter, defaultdict

from nimbus_corpus import DOCS, EVAL   # small fake company knowledge base (same folder)

print(len(DOCS), "docs |", len(EVAL), "labelled questions")
print(DOCS[0]["id"], "→", DOCS[0]["text"][:80], "...")

## 1. Analyzer: text → terms

The same function must run on documents (index time) **and** queries (search time).
Real engines use proper stemmers (Porter, Snowball); our tiny one just strips common endings, enough to see the idea.

In [ ]:
STOP_WORDS = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "is", "are", "be", "it", "my", "i",
              "do", "does", "what", "how", "can", "with", "per", "at", "by", "from", "if", "your", "you", "their"}

def stem(word: str) -> str:
    for suffix in ("ing", "ed", "es", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            word = word[: -len(suffix)]
            if suffix == "ing" and len(word) > 3 and word[-1] == word[-2]:   # resetting → resett → reset
                word = word[:-1]
            return word
    return word

def analyze(text: str) -> list[str]:
    tokens = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())   # keeps codes like e-4021 and 88-1032 together
    return [stem(t) for t in tokens if t not in STOP_WORDS]

print(analyze("Resetting the VPN passwords for E-4021 errors!"))

## 2. Inverted index: term → {doc_id: term frequency}

Like the index at the back of a textbook. At query time we only touch the lists of the query terms.

In [ ]:
class InvertedIndex:
    def __init__(self, docs: list[dict], field: str = "text"):
        self.doc_ids = [d["id"] for d in docs]
        self.doc_len: dict[str, int] = {}
        self.postings: dict[str, dict[str, int]] = defaultdict(dict)
        for d in docs:
            terms = analyze(d.get("title", "") + " " + d[field])
            self.doc_len[d["id"]] = len(terms)
            for term, tf in Counter(terms).items():
                self.postings[term][d["id"]] = tf
        self.N = len(docs)
        self.avgdl = sum(self.doc_len.values()) / self.N

index = InvertedIndex(DOCS)
print("vocabulary size:", len(index.postings), "| avg doc length:", round(index.avgdl, 1))
print("'leave' →", index.postings["leave"])
print("'e-4021' →", index.postings["e-4021"])

## 3. BM25 from scratch

```
score(D, Q) = Σ  IDF(t) × tf × (k1 + 1) / (tf + k1 × (1 − b + b × |D| / avgdl))
IDF(t)      = ln(1 + (N − n(t) + 0.5) / (n(t) + 0.5))
```

In [ ]:
class BM25:
    def __init__(self, index: InvertedIndex, k1: float = 1.2, b: float = 0.75):
        self.ix, self.k1, self.b = index, k1, b

    def idf(self, term: str) -> float:
        n = len(self.ix.postings.get(term, {}))
        return math.log(1 + (self.ix.N - n + 0.5) / (n + 0.5))

    def search(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        scores: dict[str, float] = defaultdict(float)
        for term in analyze(query):
            idf = self.idf(term)
            for doc_id, tf in self.ix.postings.get(term, {}).items():   # only docs containing the term
                norm = 1 - self.b + self.b * self.ix.doc_len[doc_id] / self.ix.avgdl
                scores[doc_id] += idf * tf * (self.k1 + 1) / (tf + self.k1 * norm)
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

bm25 = BM25(index)
for doc_id, score in bm25.search("vpn password reset"):
    print(f"{score:6.3f}  {doc_id}")

### Check: the worked example from guide 28
Three docs, query "vpn password". The guide says D1 = **1.824**, D2 = **0.591**, D3 = 0.
(We pass `title=""` and use docs that need no stemming changes.)

In [ ]:
mini = [{"id": "D1", "text": "reset vpn password"},
        {"id": "D2", "text": "vpn setup guide for vpn client on mac"},
        {"id": "D3", "text": "annual leave policy and leave carry over"}]

# The guide counts every word, so analyze without stop-word removal or stemming for this check
_saved_stop, _saved_stem = STOP_WORDS.copy(), stem
STOP_WORDS.clear(); stem = lambda w: w
print([(d, round(s, 3)) for d, s in BM25(InvertedIndex(mini)).search("vpn password")])
STOP_WORDS |= _saved_stop; stem = _saved_stem

## 4. See `k1` (saturation) and `b` (length normalisation) in numbers

**Saturation:** the TF part `tf × (k1+1) / (tf + k1)` (for an average-length doc) never exceeds `k1 + 1`.

In [ ]:
for k1 in (0.5, 1.2, 2.0):
    row = [tf * (k1 + 1) / (tf + k1) for tf in (1, 2, 3, 5, 10, 50)]
    print(f"k1={k1}: " + "  ".join(f"{v:4.2f}" for v in row), f"  (ceiling {k1 + 1})")
print("TF-IDF would be: 1, 2, 3, 5, 10, 50 ... no ceiling, keyword stuffing wins")

In [ ]:
# Length normalisation: same tf=1 match, in a short doc vs a long doc (avgdl = 20)
k1, avgdl = 1.2, 20
for b in (0.0, 0.75, 1.0):
    short = 1 * (k1 + 1) / (1 + k1 * (1 - b + b * 5 / avgdl))
    long_ = 1 * (k1 + 1) / (1 + k1 * (1 - b + b * 80 / avgdl))
    print(f"b={b}:  short doc (5 terms) {short:.2f}   long doc (80 terms) {long_:.2f}")

## 5. Evaluate BM25 on the labelled set: where does keyword search fail?

Metrics from guide 33, implemented from scratch.

In [ ]:
def hit_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    return float(any(d in relevant for d in ranked[:k]))

def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / len(relevant)

def precision_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / k

def reciprocal_rank(ranked, relevant):
    for rank, d in enumerate(ranked, start=1):
        if d in relevant:
            return 1 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    dcg = sum(1 / math.log2(r + 1) for r, d in enumerate(ranked[:k], start=1) if d in relevant)
    ideal = sum(1 / math.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

# Check against the example in guide 33: relevant {A,B}, ranked [C,A,D,B,E]
r, rel = ["C", "A", "D", "B", "E"], {"A", "B"}
print("hit@1", hit_at_k(r, rel, 1), "| recall@3", recall_at_k(r, rel, 3), "| P@5", precision_at_k(r, rel, 5),
      "| RR", reciprocal_rank(r, rel), "| nDCG@5", round(ndcg_at_k(r, rel, 5), 3))

In [ ]:
def evaluate(search_fn, k: int = 5, show_misses: bool = True) -> dict:
    rows = []
    for ex in EVAL:
        ranked = search_fn(ex["q"])
        rel = set(ex["relevant"])
        rows.append({"hit@1": hit_at_k(ranked, rel, 1), f"recall@{k}": recall_at_k(ranked, rel, k),
                     "mrr": reciprocal_rank(ranked, rel)})
        if show_misses and not hit_at_k(ranked, rel, 1):
            print(f"  NOT #1 [{ex['kind']:>10}] {ex['q']!r:55} → got {ranked[:2]}")
    return {m: round(sum(r[m] for r in rows) / len(rows), 3) for m in rows[0]}

bm25_search = lambda q: [d for d, _ in bm25.search(q, k=20)]
print("BM25:", evaluate(bm25_search))

Look at the questions BM25 didn't rank first: all **synonym** questions ("suspicious email" vs "phishing",
"maternity" vs "parental", "beer" vs "alcohol"). BM25 can't match words that aren't there. That's why we combine
it with dense retrieval ([36](36_lab_hybrid_rerank_pipeline.ipynb)).

One **exact-code** question also fails: "user got E-4120 when paying". The words "user" and "when" are **rare in our
tiny corpus** and both appear in the E-4012 doc, so their IDF adds up and beats the code itself. On a big corpus
"user"/"when" would be common (low IDF). Fixes: a better stop-word list, **boosting the title field**
(`title^2` in Elasticsearch), or a reranker. Lesson: always inspect *why* a doc scored high.

Also notice: our corpus is tiny (27 docs), so recall@5 is easy. On 1M docs, the same vocabulary gaps hurt much more.

### Cheap fix without a model: synonyms / query expansion
Search engines (Elasticsearch `synonym_graph`) let you add a synonym list. Watch the misses shrink.

In [ ]:
SYNONYMS = {"vacation": "annual leave", "holiday": "annual leave", "time off": "leave", "roll over": "carry over",
            "wifi": "broadband internet", "maternity": "parental", "beer": "alcohol meals", "maintain": "operated owner team",
            "stop paying": "cancel subscription", "suspicious email": "phishing security incident"}

def expand(q: str) -> str:
    extra = [v for k, v in SYNONYMS.items() if k in q.lower()]
    return q + " " + " ".join(extra)

print("BM25 + synonyms:", evaluate(lambda q: bm25_search(expand(q))))

Synonym lists work but **don't scale**: someone has to write them for every word. Embeddings learn this
automatically. Learned sparse models (SPLADE/ELSER) learn the expansions too.

## 6. Reciprocal Rank Fusion from scratch

Check the worked example from guide 29 (A and D should win because both lists found them).

In [ ]:
def rrf(ranked_lists: list[list[str]], k: int = 60, weights: list[float] | None = None) -> list[tuple[str, float]]:
    weights = weights or [1.0] * len(ranked_lists)
    scores: dict[str, float] = defaultdict(float)
    for w, lst in zip(weights, ranked_lists):
        for rank, doc_id in enumerate(lst, start=1):
            scores[doc_id] += w / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

bm25_list  = ["A", "B", "D"]
dense_list = ["C", "D", "E", "A"]
for doc, s in rrf([bm25_list, dense_list]):
    print(f"{doc}: {s:.5f}")

Fusion example on our corpus: combine BM25 on the **title** and BM25 on the **body** (two different
"retrievers"). In lab 36 the second list will come from embeddings.

In [ ]:
title_index = BM25(InvertedIndex([{"id": d["id"], "text": d["title"]} for d in DOCS]))
body_index = BM25(InvertedIndex([{"id": d["id"], "text": d["text"]} for d in DOCS]))

def fused_search(q):
    a = [d for d, _ in title_index.search(q, k=20)]
    b = [d for d, _ in body_index.search(q, k=20)]
    return [d for d, _ in rrf([a, b])]

print("title-only :", evaluate(lambda q: [d for d, _ in title_index.search(q, k=20)], show_misses=False))
print("body-only  :", evaluate(lambda q: [d for d, _ in body_index.search(q, k=20)], show_misses=False))
print("RRF fused  :", evaluate(fused_search, show_misses=False))

**Lesson: fusion is not magic.** The title-only retriever is weak, and fusing it in made the result *worse* than
body-only. RRF helps when both retrievers are good **and fail on different questions** (BM25 + dense, lab 36).
Always measure the fused result, never assume.

## 7. MMR: relevant **and** different

We need vectors for "similarity between docs". Without a model, we use TF-IDF vectors (bag of words).
`λ` near 1 = pure relevance, near 0 = pure diversity.

In [ ]:
def tfidf_vector(text: str) -> dict[str, float]:
    tf = Counter(analyze(text))
    return {t: c * bm25.idf(t) for t, c in tf.items()}

def cos(a: dict, b: dict) -> float:
    dot = sum(v * b.get(t, 0) for t, v in a.items())
    na, nb = math.sqrt(sum(v * v for v in a.values())), math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

def mmr(query: str, candidates: list[str], k: int = 3, lam: float = 0.5) -> list[str]:
    text = {d["id"]: d["title"] + " " + d["text"] for d in DOCS}
    qv = tfidf_vector(query)
    vecs = {c: tfidf_vector(text[c]) for c in candidates}
    selected: list[str] = []
    while len(selected) < min(k, len(candidates)):
        best = max((c for c in candidates if c not in selected),
                   key=lambda c: lam * cos(qv, vecs[c]) - (1 - lam) * max((cos(vecs[c], vecs[s]) for s in selected), default=0))
        selected.append(best)
    return selected

q = "how many leave days"
cands = [d for d, _ in bm25.search(q, k=8)]
print("plain top-3:", cands[:3])
print("MMR λ=0.7  :", mmr(q, cands, k=3, lam=0.7))
print("MMR λ=0.5  :", mmr(q, cands, k=3, lam=0.5), "  ← too much diversity, irrelevant docs sneak in")

- Plain top-3 contains **both** leave policies, v4 and the outdated v3 (near-duplicates). One context slot is wasted, and the LLM may even quote the old "20 days".
- MMR with λ=0.7 drops the duplicate and adds a *different* kind of leave (intern leave): better coverage of the question.
- λ=0.5 overdoes it: irrelevant docs get in just because they're "different". Tune λ (0.6–0.8 is common).

In real RAG you'd also **remove superseded versions** at ingestion or with a metadata filter (problem A7 in [31](31_rag_problems_and_fixes.md)).

## Revision notes
- Analyzer must be the **same** at index and query time; it decides what "a match" is.
- Inverted index = term → {doc: tf}; query time only reads the query terms' lists.
- BM25 = IDF × saturated TF with length normalisation. `k1` sets the ceiling `k1+1`; `b` punishes long docs.
- BM25 nails **exact codes**, misses **synonyms**. Synonym lists help but don't scale → embeddings / learned sparse.
- RRF = Σ w/(60 + rank): no score scaling, rewards agreement.
- Metrics: Hit@k, Recall@k, Precision@k, MRR, nDCG, all a few lines of code. **Always measure.**
- MMR trades a bit of relevance for diversity.

---
✅ **Done with 35.** Next: [36 Lab: hybrid + rerank pipeline →](36_lab_hybrid_rerank_pipeline.ipynb)